In [1]:
# ==========================================================
# Imports
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

from sklearn.calibration import CalibratedClassifierCV

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

In [2]:
# ==========================================================
# Load Dataset
# ==========================================================

MASTER = "/kaggle/input/notebooks/shri7ul/03-feature-engineering-ipynb/master_feature_engineered.parquet"

master = pd.read_parquet(MASTER)

TARGET = "is_correct"

print(master.shape)

(35072, 122)


In [3]:
# ==========================================================
# Prepare Features
# ==========================================================

DROP_COLUMNS = [

    "response_id",
    "session_id",

    "learning_objective",
    "learning_objective_id",

    "transcript",
    "student_text",
    "tutor_text",
    "background_text",

    TARGET,
]

X = master.drop(columns=DROP_COLUMNS)

y = master[TARGET].astype(int)

for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].astype("category").cat.codes

print(X.shape)

(35072, 113)


In [4]:
# ==========================================================
# Best XGBoost Parameters
# ==========================================================

BEST_PARAMS = {

    "objective":"binary:logistic",

    "eval_metric":"logloss",

    "tree_method":"hist",

    "random_state":42,

    "n_estimators":6000,

    "learning_rate":0.013793,

    "max_depth":8,

    "min_child_weight":2,

    "subsample":0.882115,

    "colsample_bytree":0.712985,

    "gamma":1.632022,

    "reg_alpha":0.461505,

    "reg_lambda":2.521299,

    "early_stopping_rounds":300,

}

In [5]:
# ==========================================================
# XGBoost + Sigmoid Calibration
# ==========================================================

kf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

oof_pred = np.zeros(len(X))
scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), 1):

    print("=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    base_model = XGBClassifier(**BEST_PARAMS)

    base_model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False,
    )

    calibrated = CalibratedClassifierCV(
        estimator=base_model,
        method="sigmoid",
        cv="prefit",
    )

    calibrated.fit(X_valid, y_valid)

    pred = calibrated.predict_proba(X_valid)[:, 1]

    oof_pred[valid_idx] = pred

    loss = log_loss(y_valid, pred)

    scores.append(loss)

    print(f"Fold LogLoss : {loss:.6f}")

print()
print("=" * 60)
print("Fold LogLoss")
print(scores)
print()
print("Mean LogLoss :", np.mean(scores))
print("Std LogLoss  :", np.std(scores))
print("=" * 60)

overall = log_loss(y, oof_pred)

print()
print("=" * 60)
print(f"Overall OOF LogLoss : {overall:.6f}")
print("=" * 60)

Fold 1
Fold LogLoss : 0.538368
Fold 2
Fold LogLoss : 0.545372
Fold 3
Fold LogLoss : 0.546086
Fold 4
Fold LogLoss : 0.545562
Fold 5
Fold LogLoss : 0.544622

Fold LogLoss
[0.5383677897314995, 0.5453717029081969, 0.5460861292005168, 0.5455617175736834, 0.5446220088020396]

Mean LogLoss : 0.5440018696431872
Std LogLoss  : 0.002855910911430742

Overall OOF LogLoss : 0.544002
